## Task 7: pairwise judging via the Kaggle model proxy
Create this as a **Kaggle Benchmarks (utility benchmark) notebook** so `kaggle_benchmarks` model calls are free.
Attach the outputs of notebooks **01, 02, 03, 04** as inputs. Judging reads saved pair files only; it never regenerates.
Resumable: if Kaggle disconnects, also attach this notebook's previous output and rerun.

In [ ]:
import os, glob, shutil, json
REPO = "/kaggle/working/small-model-alignment"
if not os.path.exists(REPO):
    src = [p for p in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True) if "/outputs/" not in p]
    assert src, "attach the repo dataset (or set up REPO via git clone)"
    shutil.copytree(os.path.dirname(src[0]), REPO, ignore=shutil.ignore_patterns("outputs"))
os.chdir(REPO)
os.makedirs("outputs/eval", exist_ok=True)
for f in glob.glob("/kaggle/input/**/outputs/eval/*_judged*", recursive=True):  # resume support
    dst = "outputs/eval/" + os.path.basename(f)
    if not os.path.exists(dst): shutil.copy(f, dst)
def find(name):
    hits = glob.glob(f"/kaggle/input/**/outputs/eval/{name}", recursive=True)
    assert hits, name
    return hits[0]
PAIRS = {k: find(f"{k}_pairs.jsonl") for k in ["dpo_hh_vs_sft", "dpo_uf_vs_sft", "lora_vs_base"]}
PAIRS

In [ ]:
import kaggle_benchmarks as kbench
try:
    print(list(kbench.llms.keys()) if hasattr(kbench.llms, "keys") else kbench.llms)
except Exception as e:
    print("could not list models:", e)
JUDGE = "google/gemini-3.6-flash"  # change if missing above; record the one you use in the report

### Dry run on 2 pairs first: check the replies parse

In [ ]:
!python scripts/judge_pairwise_kaggle.py --pairs_path "{PAIRS['dpo_uf_vs_sft']}" --output_path /tmp/dry.jsonl --rubric general --limit 2 --judge_model "{JUDGE}" --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments
!cat /tmp/dry.jsonl

In [ ]:
!python scripts/judge_pairwise_kaggle.py --pairs_path "{PAIRS['dpo_uf_vs_sft']}" --output_path outputs/eval/dpo_uf_vs_sft_judged.jsonl --rubric general --judge_model "{JUDGE}" --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments
!python scripts/judge_pairwise_kaggle.py --pairs_path "{PAIRS['dpo_hh_vs_sft']}" --output_path outputs/eval/dpo_hh_vs_sft_judged.jsonl --rubric safety --judge_model "{JUDGE}" --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments
!python scripts/judge_pairwise_kaggle.py --pairs_path "{PAIRS['lora_vs_base']}" --output_path outputs/eval/lora_vs_base_style_judged.jsonl --rubric persona --judge_model "{JUDGE}" --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments
!python scripts/judge_pairwise_kaggle.py --pairs_path "{PAIRS['lora_vs_base']}" --output_path outputs/eval/lora_vs_base_usefulness_judged.jsonl --rubric usefulness --judge_model "{JUDGE}" --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

In [ ]:
for f in sorted(glob.glob("outputs/eval/*_summary.json")):
    s = json.load(open(f))
    print(os.path.basename(f), {k: s[k] for k in ["judged", "win", "loss", "tie", "unknown", "unresolved_failures", "win_rate_excl_ties", "position_A_preference_rate"]})
!cd outputs && zip -qr ../judge_outputs.zip eval